# 01. 데이터 수집 (Data Collection)

**목표**: AI Hub 반려견 질병 말뭉치 JSON 파일을 병렬 로드하여 DataFrame으로 통합

| 항목 | 내용 |
|------|------|
| 데이터 소스 | AI Hub 반려견 성장 및 질병 관련 말뭉치 |
| 파일 형식 | JSON (파일당 1개 Q&A) |
| 주요 필드 | lifeCycle, department, disease, input, output |
| Training | 19,418개 |
| Validation | 2,427개 |

```
프로젝트 구조
pet-health-ai/
├── notebooks/
│   ├── 01_data_collection.ipynb    ← 현재
│   ├── 02_data_validation.ipynb
│   ├── 03_preprocessing.ipynb
│   ├── 04_eda.ipynb
│   ├── 05_ground_truth.ipynb
│   ├── 06_matching.ipynb
│   └── 07_evaluation.ipynb
├── utils/
│   └── config.py                   ← 경로/환경 설정
├── data/
│   ├── raw/                        ← 원본 (gitignore)
│   └── processed/                  ← 처리 결과 (gitignore)
└── docs/
    ├── aws_migration.md
    └── progress.md
```

> **AWS 마이그레이션 노트**: 현재 로컬 경로 사용 중.  
> S3 업로드 후 `.env`에 `DATA_SOURCE=s3` 설정 시 자동 전환. (`utils/config.py` 참고)

## 0. 환경 설정

In [1]:
import sys
sys.path.append('..')

import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

from utils.config import get_train_path, get_val_path, is_s3, DATA_PROCESSED

print(f"데이터 소스: {'S3' if is_s3() else '로컬'}")
print(f"Training 경로: {get_train_path()}")
print(f"Validation 경로: {get_val_path()}")

데이터 소스: 로컬
Training 경로: /Users/user/Desktop/59.반려견 성장 및 질병 관련 말뭉치 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터
Validation 경로: /Users/user/Desktop/59.반려견 성장 및 질병 관련 말뭉치 데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터


## 1. 파싱 & 병렬 로드 함수

In [2]:
def parse_record(filepath: Path) -> dict | None:
    """JSON 파일 1개를 읽어 표준 레코드 dict로 반환. 오류 시 None."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        meta = data.get('meta', {})
        qa   = data.get('qa', {})
        return {
            'lifeCycle':  meta.get('lifeCycle'),
            'department': meta.get('department'),
            'disease':    meta.get('disease'),
            'input':      qa.get('input'),
            'output':     qa.get('output'),
        }
    except Exception as e:
        print(f"[경고] {filepath.name} 파싱 실패: {e}")
        return None


def load_json_parallel(data_path: str, max_workers: int = 8) -> pd.DataFrame:
    """
    ThreadPoolExecutor로 JSON 파일을 병렬 로드.
    
    AWS 마이그레이션 후:
      - is_s3() == True 시 s3fs로 파일 목록 조회 후 동일한 parse_record 적용
    """
    if is_s3():
        # TODO: S3 마이그레이션 후 활성화
        # import s3fs
        # fs = s3fs.S3FileSystem()
        # files = [Path(f) for f in fs.glob(data_path + '*.json')]
        raise NotImplementedError("S3 로드는 AWS 셋업 후 활성화하세요. docs/aws_migration.md 참고")

    files = list(Path(data_path).rglob('*.json'))  # rglob: 하위 폴더 포함
    print(f"  발견된 파일 수: {len(files):,}개 | workers: {max_workers}")

    records = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(parse_record, f): f for f in files}
        for future in tqdm(as_completed(futures), total=len(futures), desc="로딩"):
            result = future.result()
            if result is not None:
                records.append(result)

    return pd.DataFrame(records)

## 2. Training / Validation 로드

In [3]:
print("[1/2] Training 로드")
df_train = load_json_parallel(get_train_path())
df_train['split'] = 'train'

print(f"\n[2/2] Validation 로드")
df_val = load_json_parallel(get_val_path())
df_val['split'] = 'validation'

[1/2] Training 로드
  발견된 파일 수: 19,206개 | workers: 8


로딩: 100%|██████████| 19206/19206 [00:02<00:00, 9593.73it/s] 



[2/2] Validation 로드
  발견된 파일 수: 2,400개 | workers: 8


로딩: 100%|██████████| 2400/2400 [00:00<00:00, 8002.64it/s]


## 3. 통합 DataFrame

In [4]:
df = pd.concat([df_train, df_val], ignore_index=True)

print(f"전체 레코드: {len(df):,}개")
print(f"컬럼: {list(df.columns)}")
print()
print(df['split'].value_counts().to_string())

전체 레코드: 21,606개
컬럼: ['lifeCycle', 'department', 'disease', 'input', 'output', 'split']

split
train         19206
validation     2400


## 4. 기본 구성 확인

In [5]:
print("=== lifeCycle 분포 ===")
print(df['lifeCycle'].value_counts().to_string())

print("\n=== 진료과(department) 분포 ===")
print(df['department'].value_counts().to_string())

print("\n=== 질병(disease) Top 15 ===")
print(df['disease'].value_counts().head(15).to_string())

=== lifeCycle 분포 ===
lifeCycle
성견     7276
노령견    7169
자견     7161

=== 진료과(department) 분포 ===
department
내과     11049
외과      5758
피부과     2835
안과      1085
치과       878
 치과        1

=== 질병(disease) Top 15 ===
disease
기타           16613
중성화수술          642
무릎뼈 탈구         593
중독             473
골절             408
심장사상충          396
자궁축농증          367
외이염            326
위장관폐색          256
아토피성 피부염       248
방광염            228
결막염            152
부신피질기능항진증      129
제3안검탈출증        108
유루증             89


In [6]:
# 샘플 Q&A 출력
sample = df.sample(1, random_state=42).iloc[0]
print(f"lifeCycle : {sample['lifeCycle']}")
print(f"department: {sample['department']}")
print(f"disease   : {sample['disease']}")
print(f"\n[질문]\n{sample['input']}")
print(f"\n[답변]\n{sample['output']}")

lifeCycle : 노령견
department: 내과
disease   : 기타

[질문]
5세 되는 10kg의 비숑 프리제를 기르고 있는 것으로 데, 방금 초콜릿 쿠키를 대략 10개 정도 섭취하였습니다. 초콜릿을 강아지가 섭취했을 때 어떤 종류의 건강 문제가 발생할 수 있는지 궁금합니다.

[답변]
초콜릿을 섭취할 경우 구토와 설사를 유발할 수 있습니다. 또한, 강아지의 체중에 대비한 초콜릿의 섭취량에 따라 증상이 심각해질 수 있습니다. 따라서, 반드시 동물병원을 방문하여 정밀 검사를 받으시기를 권장합니다. 씹거나 꾸준히 핥지 못하도록 부위를 보호해 주세요 불편해하면 강도는 낮춰 주세요 경과 사진을 주 1~2회 정도 남겨 주세요. 약물과 상태를 보며 보조 요법은 처방 용량을 지켜 주세요 악화되면 중단하고 진료를 받아 주세요 필요하면 간격을 조정해 주세요. 산책 상태를 보며 후 오염 부위는 미지근한 물로 가볍게 씻어 주세요 증상이 지속되면 예약 후 방문해 주세요. 일정하게 외용제는 겹겹이 바르지 않도록 주의해 주시고, 상태에 맞춰 빈도는 유연하게 조정하시고, 필요하면 계획을 조정해 주세요.


## 5. CSV 저장

In [7]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
output_path = DATA_PROCESSED / 'corpus_raw.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

size_mb = output_path.stat().st_size / 1024 / 1024
print(f"저장 완료: {output_path}")
print(f"파일 크기: {size_mb:.1f} MB")

# AWS 마이그레이션 후 S3 업로드 (docs/aws_migration.md Step 3 참고)
# if is_s3():
#     import boto3
#     s3 = boto3.client('s3', region_name=AWS_REGION)
#     s3.upload_file(str(output_path), S3_BUCKET, 'processed/corpus_raw.csv')
#     print("S3 업로드 완료")

저장 완료: /Users/user/Desktop/project/pet-health-ai/notebooks/../data/processed/corpus_raw.csv
파일 크기: 37.1 MB


## 6. 수집 요약

| 항목 | 값 |
|------|----|
| 전체 레코드 수 | (실행 후 기입) |
| Training | (실행 후 기입) |
| Validation | (실행 후 기입) |
| lifeCycle 종류 | 자견 / 성견 / 노령견 |
| 진료과 종류 | 내과 / 안과 / 외과 / 치과 / 피부과 |
| 저장 파일 | `data/processed/corpus_raw.csv` |

> **다음 단계**: `02_data_validation.ipynb` — 결측치, 중복, 클래스 불균형 검증